In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from model_utils.plots import plot_results
from lstm import LSTM
from dataset import TimeSeriesDataset
import time
import itertools
import math

In [2]:
import os
import shutil
import stat
import time

def handle_remove_readonly(func, path, exc):
    # Callback to handle read-only files on Windows
    excvalue = exc[1]
    if func in (os.rmdir, os.remove, os.unlink) and excvalue.errno == 13: # EACCES
        os.chmod(path, stat.S_IWRITE)
        func(path)
    else:
        raise

# Clean up directories from previous runs
dirs_to_cleanup = ['best_models', 'grid_search_plots', 'training_logs']
for dir_path in dirs_to_cleanup:
    if os.path.exists(dir_path):
        # Retry a few times in case of transient locks
        for i in range(3):
            try:
                shutil.rmtree(dir_path, ignore_errors=False, onerror=handle_remove_readonly)
                print(f"Removed directory: {dir_path}")
                break
            except Exception as e:
                if i < 2:
                    time.sleep(1) # Wait a bit before retrying
                else:
                    print(f"Error removing {dir_path}: {e}")

Removed directory: best_models
Removed directory: grid_search_plots
Removed directory: training_logs


In [3]:
# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../dataset/data_andre.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]

print(len(y))


# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 455
val_size = 153
forecast_horizon = 153

lookback_window = 7


df

Loading data from ../dataset/data_andre.feather...
1082371


,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,promo_value_DISC,promo_type_CIRC,promo_value_CIRC,promo_type_CIRE,promo_value_CIRE,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.0000,0,0.0000,0,0.0,0,0.0,0,0.0,6269
1,2021-01-23,154740,3,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.0000,0,0.0000,0,0.0,0,0.0,0,0.0,6269
2,2021-01-23,4978,9,juices drnks shelf stbl,pos subd grocery other,pos dept grocery,juice/aseptic/new age,0,0.0,0,...,0.0000,0,0.0000,0,0.0,0,0.0,0,0.0,6269
3,2021-01-23,904106,3,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0.1266,0,0.0000,0,0.0,0,0.0,0,0.0,6269
4,2021-01-23,904144,66,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0.0000,0,0.0000,0,0.0,0,0.0,1,0.0,6269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082366,2023-02-22,12502,3,yogurt,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.0000,0,0.0000,0,0.0,0,0.0,0,0.0,6269
1082367,2023-02-22,720252,3,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,0.0000,1,0.3718,0,0.0,0,0.0,0,0.0,6269
1082368,2023-02-22,12503,4,yogurt,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.0000,0,0.0000,0,0.0,0,0.0,0,0.0,6269
1082369,2023-02-22,151823,3,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.0000,0,0.0000,1,0.0,0,0.0,0,0.0,6269


In [4]:
print(df[TARGET_COL].describe())

count    1.082371e+06
mean     6.387899e+00
std      1.132080e+01
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      8.000000e+00
max      1.000000e+03
Name: value, dtype: float64


# Grid Search Cell

In [ ]:
import time
import itertools
import sys
import os
import importlib
sys.path.append(os.path.abspath('..'))

# Reload plot_results to pick up changes in model_utils.utils
import model_utils.plots
importlib.reload(model_utils.plots)
from model_utils.plots import plot_results

    
# Modify lstm_experiment to return timings and handle plotting internally
def lstm_experiment_grid(df, target, item_id, store_id, train_size=500, val_size=100, forecast_window=161, 
                   seq_length=30, epochs=100, batch_size=32, lr=0.001, dropout=0.0, hidden_size=32, num_layers=1, 
                   patience=50, seed=42, loss_type='MSELoss', save_plot_path=None, use_best_model=True):

    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    total_train_val = train_size + val_size
    
    total_train_val = train_size + val_size
    train_slice = slice(-(total_train_val+forecast_window), -(val_size+forecast_window))
    val_slice = slice(-(val_size+forecast_window), -forecast_window)
    test_slice = slice(-forecast_window, None)
    
    train = df[target][train_slice].values
    val = df[target][val_slice].values
    test = df[target][test_slice].values
    
    #scaler = MinMaxScaler()
    scaler = RobustScaler()
    train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()

    input_size = 1
    
    # Create Datasets
    train_dataset = TimeSeriesDataset(train_scaled, None, seq_length)
    use_pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, pin_memory=use_pin_memory)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    
    if loss_type == 'MSELoss':
        criterion = nn.MSELoss()
    elif loss_type == 'L1Loss':
        criterion = nn.L1Loss()
    elif loss_type == 'HuberLoss':
        criterion = nn.HuberLoss()
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion2 = nn.MSELoss()  # For validation loss calculation (can be different from training loss)
    
    # Validation Loader
    val_dataset = TimeSeriesDataset(val_scaled, None, seq_length)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, pin_memory=use_pin_memory)

    train_losses = []
    val_losses = []
    
    best_val_loss = float('inf')
    
    # Create directory for models if it doesn't exist
    model_dir = f'best_models/seed_{seed}/{loss_type}'
    os.makedirs(model_dir, exist_ok=True)
    best_model_path = f'{model_dir}/lstm_item{item_id}_store{store_id}.pth'
    
    # Create directory for logs
    log_dir = f'training_logs/seed_{seed}/{loss_type}'
    os.makedirs(log_dir, exist_ok=True)
    log_path = f'{log_dir}/lstm_item{item_id}_store{store_id}_log.txt'
    val_log_path = f'{log_dir}/lstm_item{item_id}_store{store_id}_val_log.txt'

    epochs_no_improve = 0
    best_epoch = 0

    # Training Time
    start_train_time = time.time()
    
    with open(log_path, 'w') as log_file, open(val_log_path, 'w') as val_log_file:
        log_file.write("Epoch,Batch,Output,Target,Loss\n")
        val_log_file.write("Epoch,Batch,Input,Output,Target,Loss\n")
        
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0
            for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                batch_x = batch_x.view(-1, seq_length, 1)

                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
            
            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            model.eval()
            all_outputs = []
            all_targets = []

            with torch.no_grad():
                for batch_idx, (batch_x, batch_y) in enumerate(val_loader):
                    batch_x = batch_x.to(device)
                    batch_y = batch_y.to(device)
                    batch_x = batch_x.view(batch_x.size(0), seq_length, 1)

                    outputs = model(batch_x)
                    all_outputs.append(outputs)
                    all_targets.append(batch_y)

                    # Log validation...

            all_outputs = torch.cat(all_outputs, dim=0).view(-1)
            all_targets = torch.cat(all_targets, dim=0).view(-1)
            
            # One loss computed over the entire validation set
            val_loss = criterion2(all_outputs, all_targets).item()
            val_losses.append(val_loss)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch + 1
                if use_best_model:
                    torch.save(model.state_dict(), best_model_path)
            
            # Optional: Print progress
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {val_loss:.6f}")

    # If NOT using best model, save the LAST model of the last epoch
    if not use_best_model:
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved last model (Epoch {epochs}) to {best_model_path}")
    else:
        print(f"Saved best model (Epoch {best_epoch}) to {best_model_path}")

    train_time = time.time() - start_train_time

    # Inference Time
    start_inference_time = time.time()
    
    # Load the model (now checking the same path where we saved the last model)
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        if use_best_model:
            print (f"Loaded model from: {best_model_path} (Best Epoch: {best_epoch})")
        else:
            print (f"Loaded model from: {best_model_path} (Last Epoch)")
    else:
        print("Warning: No model saved. Using current model in memory.")

    model.eval()
    forecast = []
    
   
    current_seq = val_scaled[-seq_length:].tolist()
    
    with torch.no_grad():
        for step in range(forecast_window):
            x = np.array(current_seq[-seq_length:]).reshape(-1, seq_length, 1) 
            x = torch.FloatTensor(x).to(device)
            pred = model(x).cpu().numpy()[0, 0]
            forecast.append(pred)
            current_seq.append(pred)
    
    forecast = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    
    inference_time = time.time() - start_inference_time
    
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    bias = np.mean(forecast - test)
    score = 0.5 * rmse + 0.25 * mae + 0.25 * abs(bias) 
    if save_plot_path:
        train_index = df[DATE_COL][train_slice].values
        val_index = df[DATE_COL][val_slice].values
        test_index = df[DATE_COL][test_slice].values
        
        plot_results(train, val, test, forecast, train_index, val_index, test_index, 
                     train_losses, val_losses, target, 
                     title=f'LSTM Forecast (Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id})',
                     save_path=save_plot_path,
                     rmse=rmse, mae=mae, bias=bias, score=score)
    
    return rmse, mae, score, bias, train_time, inference_time, best_epoch

In [6]:

# Grid Search Parameters
seeds = [42, 351, 789, 1471, 2024]
loss_functions = ['MSELoss']  
batch_size = 32
hidden_size = 32
dropout = 0.0
EPOCHS = 150  # Ensure EPOCHS is defined
LEARNING_RATE = 0.001

# Filter for specific products if needed
target_products = [916110]
if target_products:
    products = df[df['item_id'].isin(target_products)][['item_id', 'store_id']].drop_duplicates().values
else:
    # Get all unique products from the subset dataset
    products = df[['item_id', 'store_id']].drop_duplicates().values

# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)

# Run Grid Search
print(f"Starting Grid Search with {len(seeds)} seeds, {len(loss_functions)} loss functions and {len(products)} products...")

for seed in seeds:
    print(f"\n--- Processing Seed: {seed} ---")
    for loss_type in loss_functions:
        print(f"\n--- Processing Loss Type: {loss_type} ---")
        
        for item_id, store_id in products:
            print(f"Running: Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id}")
            
            # Filter data for the specific product
            df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
            
            # Handle DATE_COL (ensure it is a column and not in the index)
            if DATE_COL in df_product.index.names:
                if DATE_COL in df_product.columns:
                    # If it's in both, drop the index version to avoid "cannot insert" error
                    df_product = df_product.reset_index(drop=True)
                else:
                    # If it's only in the index, move it to a column
                    df_product = df_product.reset_index()

            # Fallback: simple reset to ensure RangeIndex 0..N
            df_product = df_product.reset_index(drop=True)

            df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
            df_product = df_product.sort_values(DATE_COL)
            df_product = df_product.reset_index(drop=True) # Final clean reset
            
            # Create directory for plots if it doesn't exist
            plot_dir = f'grid_search_plots/seed_{seed}/{loss_type}'
            os.makedirs(plot_dir, exist_ok=True)
            plot_filename = f'{plot_dir}/lstm_item{item_id}_store{store_id}.png'
            rmse, mae, bias, score, train_time, infer_time, best_epoch = lstm_experiment_grid(
                df=df_product, 
                target=TARGET_COL, 
                item_id=item_id,
                store_id=store_id,
                train_size=train_size, 
                val_size=val_size,
                forecast_window=forecast_horizon, 
                seq_length=lookback_window,
                epochs=EPOCHS,  # Use the global EPOCHS setting (e.g., 1000)
                batch_size=batch_size, 
                lr=LEARNING_RATE,
                dropout=dropout,
                hidden_size=hidden_size,
                num_layers=1,
                patience=1000,
                seed=seed,
                loss_type=loss_type,
                save_plot_path=plot_filename
            )
            
            results.append({
                'seed': seed,
                'loss_type': loss_type,
                'item_id': item_id,
                'store_id': store_id,
                'batch_size': batch_size,
                'hidden_size': hidden_size,
                'dropout': dropout,
                'rmse': rmse,
                'mae': mae,
                'bias': bias,
                'score': score,
                'train_time': train_time,
                'inference_time': infer_time,
                'best_epoch': best_epoch,
                'plot_path': plot_filename
            })
        

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.to_csv('grid_search_results.csv', index=False)

Starting Grid Search with 5 seeds, 1 loss functions and 1 products...

--- Processing Seed: 42 ---

--- Processing Loss Type: MSELoss ---
Running: Seed=42, Loss=MSELoss, Item=916110, Store=6269
Epoch 10/150 | Train Loss: 0.572555 | Val Loss: 0.591498
Epoch 20/150 | Train Loss: 0.498538 | Val Loss: 0.471885
Epoch 30/150 | Train Loss: 0.468346 | Val Loss: 0.488629
Epoch 40/150 | Train Loss: 0.461942 | Val Loss: 0.496874
Epoch 50/150 | Train Loss: 0.457820 | Val Loss: 0.500461
Epoch 60/150 | Train Loss: 0.454402 | Val Loss: 0.503597
Epoch 70/150 | Train Loss: 0.451214 | Val Loss: 0.506384
Epoch 80/150 | Train Loss: 0.447958 | Val Loss: 0.509201
Epoch 90/150 | Train Loss: 0.444442 | Val Loss: 0.512152
Epoch 100/150 | Train Loss: 0.440483 | Val Loss: 0.515240
Epoch 110/150 | Train Loss: 0.435829 | Val Loss: 0.518532
Epoch 120/150 | Train Loss: 0.430099 | Val Loss: 0.522262
Epoch 130/150 | Train Loss: 0.422855 | Val Loss: 0.526750
Epoch 140/150 | Train Loss: 0.414091 | Val Loss: 0.531797
Epo